In [ ]:
# PUBLIC REPOSITORY NOTE: Do not publish patient-level CSV exports or notebook outputs.
# Study/record identifiers are used only for internal validation and alignment.


# PAFA Postmortem Study
# Analysis started: November 21, 2026

import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load the latest dataset
file = "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v03.csv"
df = pd.read_csv(file)

# Initial inspection
print("Dataset dimensions (rows, columns):", df.shape)
print("Dataset loaded. Individual patient rows are intentionally not displayed.")


In [ ]:
# Remove original raw clinical variables
start = df.columns.get_loc("young_onset_parkinsonism.1")
end = df.columns.get_loc("other_atypical.4")

df_v04 = df.drop(columns=df.columns[start:end + 1])

# Save as version 04 (LOCAL PATIENT-LEVEL DATA; NEVER UPLOAD CSV TO GITHUB)
df_v04.to_csv(
    "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v04.csv",
    index=False
)

# Verify
print("Original shape:", df.shape)
print("New shape:", df_v04.shape)
print("Columns removed:", df.shape[1] - df_v04.shape[1])

In [ ]:
# Explore the new PAFA dataset (v04)

print("Dataset dimensions:", df_v04.shape)

print("\nColumn names:")
print([c for c in df_v04.columns if c != "record_id"])

print("Individual patient rows are intentionally not displayed.")

print("\nMissing values:")
print(df_v04.isna().sum()[df_v04.isna().sum() > 0])

print("\nNumber of unique subjects:")
print(df_v04["record_id"].nunique())

In [ ]:
# Count subjects for each primary pathological diagnosis (codes 0–8)

df_v04["primary_parkinsonism"].value_counts().reindex(
    range(9), fill_value=0
).sort_index()

In [ ]:
# Create version 05 (LOCAL PATIENT-LEVEL DATA; NEVER UPLOAD CSV TO GITHUB)
df_v05 = df_v04.copy()

# Recode 4 and 5 as 9 (Lewy body disease)
df_v05["primary_parkinsonism"] = (
    df_v05["primary_parkinsonism"].replace({4: 9, 5: 9})
)

# Verify counts
print(df_v05["primary_parkinsonism"].value_counts().sort_index())

# Export version 05
df_v05.to_csv(
    "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v05.csv",
    index=False
)

In [ ]:
# Find all recoded PAFA question columns for the three doctors
pafa_columns = [
    col for col in df_v05.columns
    if "_rcd." in col and col.endswith((".1", ".2", ".4"))
]

print("Number of columns:", len(pafa_columns))
print("Number of PAFA questions:", len(pafa_columns) // 3)

# Display the columns
print("\n".join(pafa_columns))

In [ ]:
import pandas as pd
import numpy as np

# Work from version 05
df_check = df_v05.copy()

# Identify the 36 recoded PAFA questions
questions = sorted(set(
    col.rsplit(".", 1)[0]
    for col in df_check.columns
    if "_rcd." in col and col.endswith((".1", ".2", ".4"))
))

# Calculate the median response for each question
for q in questions:
    cols = [f"{q}.{i}" for i in [1, 2, 4]]
    df_check[f"{q}_median"] = df_check[cols].median(axis=1)

median_cols = [f"{q}_median" for q in questions]

# Method A: Sum of the 36 question medians
df_check["sum_question_medians"] = df_check[median_cols].sum(axis=1)

# Method B: Median of the 3 physicians' total PAFA scores
df_check["median_physician_totals"] = df_check[
    ["PAFA_tot.1", "PAFA_tot.2", "PAFA_tot.4"]
].median(axis=1)

# Compare methods
df_check["difference"] = (
    df_check["sum_question_medians"]
    - df_check["median_physician_totals"]
)

print("Number of PAFA questions:", len(questions))
print("Number of patients:", len(df_check))

print("\nNumber of patients with different results:")
print((df_check["difference"] != 0).sum())

print("\nSummary of differences:")
print(df_check["difference"].describe())

# Patient-level comparison omitted from public notebook; aggregate summaries preserved.


In [ ]:
# Compare mean vs. median total PAFA scores

df_compare = df_v05.copy()

# Three physicians' total PAFA scores
score_cols = ["PAFA_tot.1", "PAFA_tot.2", "PAFA_tot.4"]

# Calculate mean and median
df_compare["PAFA_mean"] = df_compare[score_cols].mean(axis=1)
df_compare["PAFA_median"] = df_compare[score_cols].median(axis=1)

# Calculate difference
df_compare["difference"] = (
    df_compare["PAFA_mean"] - df_compare["PAFA_median"]
)

# Summary statistics
print("Summary statistics:")
display(
    df_compare[["PAFA_mean", "PAFA_median", "difference"]].describe()
)

# Number of patients with different mean and median scores
print("\nPatients with different mean and median scores:")
print((df_compare["difference"].abs() > 1e-8).sum())

# Correlation between methods
print("\nPearson correlation:")
print(df_compare["PAFA_mean"].corr(df_compare["PAFA_median"]))

print("\nSpearman correlation:")
print(
    df_compare["PAFA_mean"].corr(
        df_compare["PAFA_median"],
        method="spearman"
    )
)

# Patient-level comparison
# Patient-level comparison omitted from public notebook; aggregate summaries preserved.


In [ ]:
import pandas as pd

# Load version 05 from the current folder
df_v05 = pd.read_csv(
    "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v05.csv"
)

# Identify the 36 recoded PAFA questions
questions = list(dict.fromkeys(
    col.rsplit(".", 1)[0]
    for col in df_v05.columns
    if "_rcd." in col and col.endswith((".1", ".2", ".4"))
))

# Verify all three physician ratings are present
doctor_cols = [
    f"{q}.{doctor}"
    for q in questions
    for doctor in [1, 2, 4]
]

assert len(questions) == 36
assert len(doctor_cols) == 108
assert all(col in df_v05.columns for col in doctor_cols)

# Check that all recoded responses are binary
assert df_v05[doctor_cols].isin([0, 1]).all().all()

# Create version 06
df_v06 = df_v05.copy()

# Calculate median for each individual PAFA question
median_cols = []

for q in questions:
    new_col = f"{q}_median"

    df_v06[new_col] = df_v06[
        [f"{q}.1", f"{q}.2", f"{q}.4"]
    ].median(axis=1)

    median_cols.append(new_col)

# Calculate mean of the three physician scores
score_groups = {
    "pafa_tot_mean": ["PAFA_tot.1", "PAFA_tot.2", "PAFA_tot.4"],
    "pafa_motor_mean": ["PAFA_motor.1", "PAFA_motor.2", "PAFA_motor.4"],
    "pafa_nonmotor_mean": ["PAFA_nonmotor.1", "PAFA_nonmotor.2", "PAFA_nonmotor.4"]
}

for new_col, cols in score_groups.items():
    df_v06[new_col] = df_v06[cols].mean(axis=1)

# Remove the 108 individual physician question columns
df_v06 = df_v06.drop(columns=doctor_cols)

# Confirm pathological diagnosis coding
assert not df_v06["primary_parkinsonism"].isin([4, 5]).any()
assert df_v06["record_id"].is_unique

# Export version 06 (LOCAL PATIENT-LEVEL DATA; NEVER UPLOAD CSV TO GITHUB)
filename = "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v06.csv"
df_v06.to_csv(filename, index=False)

# ============================================================
# OUTPUT TO COPY AND SHARE
# ============================================================

print("DATASET VERSION 06 VERIFICATION")
print("--------------------------------")

print("Number of patients:", len(df_v06))
print("Number of columns:", df_v06.shape[1])
print("Number of median PAFA questions:", len(median_cols))
print("Number of individual physician question columns remaining:",
      sum(col in df_v06.columns for col in doctor_cols))

print("\nPRIMARY PATHOLOGICAL DIAGNOSES:")
print(df_v06["primary_parkinsonism"].value_counts().sort_index())

print("\nCLINICAL DIAGNOSES:")
print(df_v06["clinical_diagnosis"].value_counts().sort_index())

print("\nPAFA MEAN SCORE SUMMARY:")
print(
    df_v06[
        ["pafa_tot_mean", "pafa_motor_mean", "pafa_nonmotor_mean"]
    ].describe().to_string()
)

print("\nMISSING VALUES IN ANALYSIS VARIABLES:")
analysis_cols = (
    ["clinical_diagnosis", "primary_parkinsonism"]
    + median_cols
    + ["pafa_tot_mean", "pafa_motor_mean", "pafa_nonmotor_mean"]
)

print(df_v06[analysis_cols].isna().sum().loc[lambda x: x > 0])

print("\nDUPLICATE PATIENT IDs:")
print(df_v06["record_id"].duplicated().sum())

print("\nEXPORTED FILE:")
print(filename)

In [ ]:
# Compare original PAFA mean with our calculated mean

original = df_v05["pafa_tot_mean"]

calculated = df_v05[
    ["PAFA_tot.1", "PAFA_tot.2", "PAFA_tot.4"]
].mean(axis=1)

print("Pearson correlation:", original.corr(calculated))

print("Maximum absolute difference:",
      (original - calculated).abs().max())

print("Number of patients with differences:",
      ((original - calculated).abs() > 1e-8).sum())

In [ ]:

import pandas as pd

# Load existing version 06
df_v06 = pd.read_csv(
    "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v06.csv"
)

# Identify the 36 existing median PAFA questions
median_cols = [
    col for col in df_v06.columns
    if col.endswith("_rcd_median")
]

# Select only the variables needed for our heatmaps
keep_cols = (
    ["record_id"]
    + median_cols
    + [
        "pafa_tot_mean",
        "pafa_motor_mean",
        "pafa_nonmotor_mean",
        "clinical_diagnosis",
        "primary_parkinsonism"
    ]
)

# Create clean analysis dataset
df_v07 = df_v06[keep_cols].copy()

# Verify structure
assert len(median_cols) == 36
assert df_v07["record_id"].is_unique
assert not df_v07.columns.duplicated().any()

print("Patients:", len(df_v07))
print("Columns:", len(df_v07.columns))
print("Median PAFA questions:", len(median_cols))

print("\nPrimary pathological diagnoses:")
print(df_v07["primary_parkinsonism"].value_counts().sort_index())

print("\nClinical diagnoses:")
print(df_v07["clinical_diagnosis"].value_counts().sort_index())

print("\nMissing values:")
print(df_v07.isna().sum().sum())

# Export version 07 (LOCAL PATIENT-LEVEL DATA; NEVER UPLOAD CSV TO GITHUB)
filename = "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v07.csv"
df_v07.to_csv(filename, index=False)

print("\nExported:", filename)


In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v07.csv")

# Output folder
outdir = "PAFA_heatmaps_v07"
os.makedirs(outdir, exist_ok=True)

# ============================================================
# EXACT PAFA QUESTION ORDER + Y-AXIS LABELS
# ============================================================

question_cols = [
    "young_onset_parkinsonism_rcd_median",
    "dementia_early_rcd_median",
    "levopdopa_responsive_rcd_median",
    "no_rest_tremor_rcd_median",
    "cervical_dystonia_rcd_median",
    "face_jaw_dystonia_rcd_median",
    "hemidystonia_rcd_median",
    "myoclonus_rcd_median",
    "camptocormia_rcd_median",
    "limb_ataxia_rcd_median",
    "gait_ataxia_rcd_median",
    "early_falls_rcd_median",
    "early_fog_rcd_median",
    "early_wheelchair_rcd_median",
    "lmn_signs_rcd_median",
    "umn_signs_rcd_median",
    "pseudobulbar_rcd_median",
    "constipation_severe_rcd_median",
    "erectile_dysfunction_rcd_median",
    "gi_dysmotility_rcd_median",
    "orthostatic_hypotension_rcd_median",
    "urinary_urgency_rcd_median",
    "dysarthria_rcd_median",
    "dysphagia_rcd_median",
    "gravelly_voice_rcd_median",
    "supranuclear_palsy_rcd_median",
    "vocal_cord_probs_rcd_median",
    "alien_limb_rcd_median",
    "apraxia_rcd_median",
    "cortical_sensory_loss_rcd_median",
    "language_disturbance_rcd_median",
    "visuospatial_probs_rcd_median",
    "apathy_rcd_median",
    "behavioral_change_rcd_median",
    "psychosis_rcd_median",
    "other_atypical_rcd_median",
]

question_labels = [
    "Young onset",
    "Early dementia",
    "No response to levodopa",
    "No rest tremor",
    "Cervical dystonia",
    "Face or jaw dystonia",
    "Hemidystonia",
    "Myoclonus",
    "Camptocormia or head drop",
    "Cerebellar limb ataxia",
    "Cerebellar gait ataxia",
    "Early falls",
    "Early freezing of gait",
    "Early wheelchair",
    "Lower motor neuron signs",
    "Upper motor neuron signs",
    "Pseudobulbar affect",
    "Severe constipation",
    "Erectile dysfunction or anorgasmia",
    "Gastrointestinal dysmotility",
    "Severe orthostatic hypotension",
    "Urinary urgency or incontinence",
    "Severe dysarthria",
    "Severe dysphagia",
    "Gravelly voice or festinating speech",
    "Supranuclear gaze palsy",
    "Vocal cord / laryngeal dysfunction",
    "Alien limb",
    "Apraxia",
    "Cortical sensory loss",
    "Language disturbance",
    "Visuospatial impairment",
    "Apathy",
    "Behavioral changes",
    "Hallucinations / psychosis",
    "Other atypical features",
]

score_cols = ["pafa_tot_mean", "pafa_motor_mean", "pafa_nonmotor_mean"]

# ============================================================
# DIAGNOSIS LABELS
# ============================================================

clinical_map = {
    1: "Parkinson's Disease",
    2: "Atypical Parkinsonism (Unspecified)",
    3: "Frontotemporal Dementia",
    4: "Lewy Body Disease",
    5: "Multiple System Atrophy",
    6: "Progressive Supranuclear Palsy",
    8: "Corticobasal Degeneration",
}

path_map = {
    2: "Frontotemporal Dementia - TDP-43",
    3: "Multiple System Atrophy",
    6: "Progressive Supranuclear Palsy",
    9: "Lewy Body Disease",
}

# Keep only diagnoses actually present
clinical_codes = [c for c in clinical_map if c in df["clinical_diagnosis"].unique()]
path_codes = [c for c in path_map if c in df["primary_parkinsonism"].unique()]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_pearson(x, y):
    x = pd.Series(x)
    y = pd.Series(y)
    if x.nunique(dropna=True) < 2 or y.nunique(dropna=True) < 2:
        return np.nan, np.nan
    r, p = pearsonr(x, y)
    return r, p

def clean_r(x):
    if pd.isna(x):
        return np.nan
    if abs(x) < 0.005:
        return 0.0
    return x

def build_corr_matrices(data, feature_cols, diag_col, diag_codes, diag_map):
    rmat = pd.DataFrame(index=feature_cols, columns=diag_codes, dtype=float)
    pmat = pd.DataFrame(index=feature_cols, columns=diag_codes, dtype=float)

    for code in diag_codes:
        indicator = (data[diag_col] == code).astype(int)
        for feature in feature_cols:
            r, p = safe_pearson(data[feature], indicator)
            rmat.loc[feature, code] = r
            pmat.loc[feature, code] = p

    # Benjamini-Hochberg adjustment across all cells in this figure
    valid_mask = pmat.notna().values.ravel()
    pvals = pmat.values.ravel()[valid_mask]
    adj = np.full(pmat.size, np.nan)

    if len(pvals) > 0:
        _, pvals_adj, _, _ = multipletests(pvals, method="fdr_bh")
        adj[valid_mask] = pvals_adj

    pmat_adj = pd.DataFrame(
        adj.reshape(pmat.shape),
        index=pmat.index,
        columns=pmat.columns
    )

    # Order columns by mean Pearson r (most negative to most positive)
    col_order = rmat.mean(axis=0).sort_values().index.tolist()

    rmat = rmat[col_order]
    pmat = pmat[col_order]
    pmat_adj = pmat_adj[col_order]

    return rmat, pmat, pmat_adj, col_order

def make_annotations(rmat, pmat, pmat_adj):
    ann = pd.DataFrame(index=rmat.index, columns=rmat.columns, dtype=object)

    for i in rmat.index:
        for j in rmat.columns:
            r = rmat.loc[i, j]
            p = pmat.loc[i, j]
            p_adj = pmat_adj.loc[i, j]

            if pd.isna(r):
                ann.loc[i, j] = ""
                continue

            r = clean_r(r)

            if pd.notna(p_adj) and p_adj < 0.05:
                star = "**"
            elif pd.notna(p) and p < 0.05:
                star = "*"
            else:
                star = ""

            ann.loc[i, j] = f"{r:.2f}{star}"

    return ann

def wrapped_label(s, width=18):
    return "\n".join(textwrap.wrap(s, width=width, break_long_words=False))

def diagnosis_labels_with_n(data, diag_col, col_order, diag_map, wrap_width=18):
    labels = []
    for code in col_order:
        n = int((data[diag_col] == code).sum())
        base = diag_map[code]
        label = f"{wrapped_label(base, width=wrap_width)}\n(n={n})"
        labels.append(label)
    return labels

def draw_heatmap(
    rmat,
    ann,
    xlabels,
    ylabels,
    title,
    filename_base,
    xlabel,
    ylabel,
    footnote_main,
    footnote_extra=None,
    figsize=(16, 14),
    title_size=20,
    annot_size=9,
    xtick_size=10,
    ytick_size=10,
):
    sns.set_style("white")
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("#f2f2f2")
    ax.set_facecolor("#f2f2f2")

    hm = sns.heatmap(
        rmat,
        cmap="RdBu",
        vmin=-1,
        vmax=1,
        center=0,
        annot=ann,
        fmt="",
        linewidths=0.8,
        linecolor="#d9d9d9",
        cbar=True,
        cbar_kws={"label": "Pearson correlation coefficient (r)"},
        annot_kws={"fontsize": annot_size},
        ax=ax,
    )

    ax.set_title(title, fontsize=title_size, pad=14, weight="bold")
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)

    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=xtick_size)
    ax.set_yticklabels(ylabels, rotation=0, fontsize=ytick_size)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("Pearson correlation coefficient (r)", fontsize=11)

    footnote = footnote_main
    if footnote_extra:
        footnote = footnote + "\n" + footnote_extra

    plt.figtext(
        0.5,
        0.015,
        footnote,
        ha="center",
        va="bottom",
        fontsize=10
    )

    plt.tight_layout(rect=[0, 0.06, 1, 0.98])

    png_path = os.path.join(outdir, f"{filename_base}.png")
    pdf_path = os.path.join(outdir, f"{filename_base}.pdf")

    plt.savefig(png_path, dpi=600, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.savefig(pdf_path, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

def single_score_heatmap(
    data,
    score_col,
    score_label,
    diag_col,
    diag_codes,
    diag_map,
    title,
    filename_base,
    xlabel="Diagnosis",
    ylabel="PAFA score",
):
    rmat, pmat, pmat_adj, col_order = build_corr_matrices(
        data=data,
        feature_cols=[score_col],
        diag_col=diag_col,
        diag_codes=diag_codes,
        diag_map=diag_map
    )

    ann = make_annotations(rmat, pmat, pmat_adj)
    xlabels = diagnosis_labels_with_n(data, diag_col, col_order, diag_map, wrap_width=18)
    ylabels = [score_label]

    footnote_main = (
        "* Unadjusted p < 0.05   |   ** Benjamini-Hochberg adjusted p < 0.05"
    )
    footnote_extra = (
        "PAFA summary scores represent the mean of three independent evaluator scores.\n"
        "Columns ordered by mean Pearson r (most negative to most positive). "
        "Interpret small groups cautiously (n = 1-3)."
    )

    draw_heatmap(
        rmat=rmat,
        ann=ann,
        xlabels=xlabels,
        ylabels=ylabels,
        title=title,
        filename_base=filename_base,
        xlabel=xlabel,
        ylabel=ylabel,
        footnote_main=footnote_main,
        footnote_extra=footnote_extra,
        figsize=(14, 4.5),
        title_size=18,
        annot_size=14,
        xtick_size=10,
        ytick_size=12,
    )

# ============================================================
# QUESTION HEATMAPS
# ============================================================

# Clinical diagnoses
r_clin_q, p_clin_q, padj_clin_q, order_clin_q = build_corr_matrices(
    data=df,
    feature_cols=question_cols,
    diag_col="clinical_diagnosis",
    diag_codes=clinical_codes,
    diag_map=clinical_map
)

ann_clin_q = make_annotations(r_clin_q, p_clin_q, padj_clin_q)
xlabels_clin_q = diagnosis_labels_with_n(df, "clinical_diagnosis", order_clin_q, clinical_map, wrap_width=20)

draw_heatmap(
    rmat=r_clin_q,
    ann=ann_clin_q,
    xlabels=xlabels_clin_q,
    ylabels=question_labels,
    title="PAFA Questions vs. Clinical Diagnoses",
    filename_base="PAFA_questions_vs_clinical_diagnoses",
    xlabel="Diagnosis",
    ylabel="PAFA Question",
    footnote_main="* Unadjusted p < 0.05   |   ** Benjamini-Hochberg adjusted p < 0.05",
    footnote_extra="Columns ordered by mean Pearson r (most negative to most positive). "
                   "Interpret small groups cautiously (n = 1-3).",
    figsize=(18, 15),
    title_size=20,
    annot_size=8,
    xtick_size=10,
    ytick_size=10,
)

# Pathological diagnoses
r_path_q, p_path_q, padj_path_q, order_path_q = build_corr_matrices(
    data=df,
    feature_cols=question_cols,
    diag_col="primary_parkinsonism",
    diag_codes=path_codes,
    diag_map=path_map
)

ann_path_q = make_annotations(r_path_q, p_path_q, padj_path_q)
xlabels_path_q = diagnosis_labels_with_n(df, "primary_parkinsonism", order_path_q, path_map, wrap_width=20)

draw_heatmap(
    rmat=r_path_q,
    ann=ann_path_q,
    xlabels=xlabels_path_q,
    ylabels=question_labels,
    title="PAFA Questions vs. Pathological Diagnoses",
    filename_base="PAFA_questions_vs_pathological_diagnoses",
    xlabel="Diagnosis",
    ylabel="PAFA Question",
    footnote_main="* Unadjusted p < 0.05   |   ** Benjamini-Hochberg adjusted p < 0.05",
    footnote_extra="Columns ordered by mean Pearson r (most negative to most positive). "
                   "Interpret small groups cautiously (n = 1-3).",
    figsize=(16, 15),
    title_size=20,
    annot_size=8,
    xtick_size=10,
    ytick_size=10,
)

# ============================================================
# SEPARATE SUMMARY-SCORE HEATMAPS
# ============================================================

# Total score
single_score_heatmap(
    data=df,
    score_col="pafa_tot_mean",
    score_label="Total PAFA mean",
    diag_col="clinical_diagnosis",
    diag_codes=clinical_codes,
    diag_map=clinical_map,
    title="Total PAFA Mean Score vs. Clinical Diagnoses",
    filename_base="PAFA_total_mean_vs_clinical_diagnoses"
)

single_score_heatmap(
    data=df,
    score_col="pafa_tot_mean",
    score_label="Total PAFA mean",
    diag_col="primary_parkinsonism",
    diag_codes=path_codes,
    diag_map=path_map,
    title="Total PAFA Mean Score vs. Pathological Diagnoses",
    filename_base="PAFA_total_mean_vs_pathological_diagnoses"
)

# Motor score
single_score_heatmap(
    data=df,
    score_col="pafa_motor_mean",
    score_label="Motor PAFA mean",
    diag_col="clinical_diagnosis",
    diag_codes=clinical_codes,
    diag_map=clinical_map,
    title="Motor PAFA Mean Score vs. Clinical Diagnoses",
    filename_base="PAFA_motor_mean_vs_clinical_diagnoses"
)

single_score_heatmap(
    data=df,
    score_col="pafa_motor_mean",
    score_label="Motor PAFA mean",
    diag_col="primary_parkinsonism",
    diag_codes=path_codes,
    diag_map=path_map,
    title="Motor PAFA Mean Score vs. Pathological Diagnoses",
    filename_base="PAFA_motor_mean_vs_pathological_diagnoses"
)

# Nonmotor score
single_score_heatmap(
    data=df,
    score_col="pafa_nonmotor_mean",
    score_label="Nonmotor PAFA mean",
    diag_col="clinical_diagnosis",
    diag_codes=clinical_codes,
    diag_map=clinical_map,
    title="Nonmotor PAFA Mean Score vs. Clinical Diagnoses",
    filename_base="PAFA_nonmotor_mean_vs_clinical_diagnoses"
)

single_score_heatmap(
    data=df,
    score_col="pafa_nonmotor_mean",
    score_label="Nonmotor PAFA mean",
    diag_col="primary_parkinsonism",
    diag_codes=path_codes,
    diag_map=path_map,
    title="Nonmotor PAFA Mean Score vs. Pathological Diagnoses",
    filename_base="PAFA_nonmotor_mean_vs_pathological_diagnoses"
)

# ============================================================
# EXPORT SUMMARY
# ============================================================

print("Heatmaps exported to folder:", outdir)
print("\nFiles created:")
for f in sorted(os.listdir(outdir)):
    print(" -", f)

In [ ]:

# ============================================================
# PAFA STUDY — HEATMAPS VERSION 08
# 10 publication-quality PNG heatmaps, 600 DPI
# ============================================================

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
from matplotlib import font_manager

# ============================================================
# 1. LOAD DATASETS
# ============================================================

df = pd.read_csv(
    "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v07.csv"
)

original = pd.read_csv(
    "AllPAFA_Clinical_OrigOnset_2026-11-18_wide_v05.csv"
)

folder = "PAFA_heatmaps_v08"
os.makedirs(folder, exist_ok=True)

assert df["record_id"].is_unique
assert original["record_id"].is_unique
assert set(df["record_id"]) == set(original["record_id"])

original = original.set_index("record_id").loc[df["record_id"]]
original = original.reset_index()

print("Patients:", len(df))

# ============================================================
# 2. VISUAL STYLE
# ============================================================

available_fonts = {
    f.name for f in font_manager.fontManager.ttflist
}

font = next(
    (f for f in ["Helvetica", "Arial", "Liberation Sans"]
     if f in available_fonts),
    "DejaVu Sans"
)

plt.rcParams.update({
    "font.family": font,
    "font.size": 11,
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "savefig.dpi": 600,
    "figure.facecolor": "white"
})

sns.set_style("white")

# ============================================================
# 3. ALL 36 PAFA QUESTIONS
# Researcher's approved wording and order
# ============================================================

items = [
    ("young_onset_parkinsonism", "Young onset"),
    ("dementia_early", "Early dementia"),
    ("levopdopa_responsive", "No response to levodopa"),
    ("no_rest_tremor", "No rest tremor"),
    ("cervical_dystonia", "Cervical dystonia"),
    ("face_jaw_dystonia", "Face or jaw dystonia"),
    ("hemidystonia", "Hemidystonia"),
    ("myoclonus", "Myoclonus"),
    ("camptocormia", "Camptocormia or head drop"),
    ("limb_ataxia", "Cerebellar limb ataxia"),
    ("gait_ataxia", "Cerebellar gait ataxia"),
    ("early_falls", "Early falls"),
    ("early_fog", "Early freezing of gait"),
    ("early_wheelchair", "Early wheelchair"),
    ("lmn_signs", "Lower motor neuron signs"),
    ("umn_signs", "Upper motor neuron signs"),
    ("pseudobulbar", "Pseudobulbar affect"),
    ("constipation_severe", "Severe constipation"),
    ("erectile_dysfunction", "Erectile dysfunction or anorgasmia"),
    ("gi_dysmotility", "Gastrointestinal dysmotility"),
    ("orthostatic_hypotension", "Severe orthostatic hypotension"),
    ("urinary_urgency", "Urinary urgency or urge incontinence"),
    ("dysarthria", "Severe dysarthria"),
    ("dysphagia", "Severe dysphagia"),
    ("gravelly_voice", "Gravelly voice or festinating speech"),
    ("supranuclear_palsy", "Supranuclear gaze palsy"),
    ("vocal_cord_probs", "Vocal cord / laryngeal dysfunction"),
    ("alien_limb", "Alien limb"),
    ("apraxia", "Apraxia"),
    ("cortical_sensory_loss", "Cortical sensory loss"),
    ("language_disturbance", "Language disturbance"),
    ("visuospatial_probs", "Visuospatial impairment"),
    ("apathy", "Apathy"),
    ("behavioral_change", "Behavioral changes"),
    ("psychosis", "Hallucinations / psychosis"),
    ("other_atypical", "Other atypical features")
]

question_cols = [
    f"{name}_rcd_median" for name, label in items
]

question_labels = [label for name, label in items]

assert len(question_cols) == 36
assert all(c in df.columns for c in question_cols)

# ============================================================
# 4. DIAGNOSIS DEFINITIONS
# ============================================================

clinical = {
    1: "Parkinson's Disease",
    2: "Atypical Parkinsonism (Unspecified)",
    3: "Frontotemporal Dementia",
    4: "Lewy Body Disease",
    5: "Multiple System Atrophy",
    6: "Progressive Supranuclear Palsy",
    8: "Corticobasal Degeneration"
}

pathological = {
    2: "Frontotemporal Dementia - TDP-43",
    3: "Multiple System Atrophy",
    6: "Progressive Supranuclear Palsy",
    9: "Lewy Body Disease"
}

diagnoses = {
    "Clinical": ("clinical_diagnosis", clinical),
    "Pathological": ("primary_parkinsonism", pathological)
}

assert set(df["clinical_diagnosis"]) == set(clinical)
assert set(df["primary_parkinsonism"]) == set(pathological)

# ============================================================
# 5. NINE PAFA DOMAINS
# ============================================================

domains = {
    "Disease Time Course": [
        "young_onset_parkinsonism",
        "dementia_early"
    ],
    "Levodopa-unresponsiveness": [
        "levopdopa_responsive"
    ],
    "Hyperkinetic Features": [
        "no_rest_tremor",
        "cervical_dystonia",
        "face_jaw_dystonia",
        "hemidystonia",
        "myoclonus"
    ],
    "Posture, Gait, and Balance": [
        "camptocormia",
        "limb_ataxia",
        "gait_ataxia",
        "early_falls",
        "early_fog",
        "early_wheelchair"
    ],
    "Pyramidal Signs": [
        "lmn_signs",
        "umn_signs",
        "pseudobulbar"
    ],
    "Autonomic Features": [
        "constipation_severe",
        "erectile_dysfunction",
        "gi_dysmotility",
        "orthostatic_hypotension",
        "urinary_urgency"
    ],
    "Bulbar Signs and Eye Movements": [
        "dysarthria",
        "dysphagia",
        "gravelly_voice",
        "supranuclear_palsy",
        "vocal_cord_probs"
    ],
    "Cognitive Signs / Corticobasal Syndrome": [
        "alien_limb",
        "apraxia",
        "cortical_sensory_loss",
        "language_disturbance",
        "visuospatial_probs"
    ],
    "Psychiatric Symptoms": [
        "apathy",
        "behavioral_change",
        "psychosis"
    ]
}

domain_labels = list(domains)

# 35 assigned questions; Other atypical is excluded
assigned = [item for group in domains.values() for item in group]

assert len(domains) == 9
assert len(assigned) == 35
assert len(set(assigned)) == 35
assert set(assigned) == (
    {name for name, label in items} - {"other_atypical"}
)

# ============================================================
# 6. CALCULATE DOMAIN MEANS FROM ORIGINAL PHYSICIAN DATA
# ============================================================

subscore_names = {
    "Hyperkinetic Features": "hyperkinetic_subscore",
    "Posture, Gait, and Balance": "pos_gait_balance_subscore",
    "Pyramidal Signs": "pyramidal_subscore",
    "Autonomic Features": "autonomic_subscore",
    "Bulbar Signs and Eye Movements": "bulbar_eye_subscore",
    "Cognitive Signs / Corticobasal Syndrome": "cognitive_subscore",
    "Psychiatric Symptoms": "psychiatric_subscore"
}

domain_cols = []

for number, (domain, features) in enumerate(domains.items(), 1):

    output = f"domain_{number}"
    domain_cols.append(output)

    if domain in subscore_names:

        prefix = subscore_names[domain]

        columns = [
            f"{prefix}.{doctor}" for doctor in [1, 2, 4]
        ]

        assert all(c in original.columns for c in columns)
        assert original[columns].notna().all().all()

        df[output] = original[columns].mean(axis=1).to_numpy()

    else:

        physician_scores = []

        for doctor in [1, 2, 4]:

            columns = [
                f"{feature}_rcd.{doctor}"
                for feature in features
            ]

            assert all(c in original.columns for c in columns)
            assert original[columns].isin([0, 1]).all().all()

            physician_scores.append(
                original[columns].sum(axis=1).to_numpy()
            )

        df[output] = np.mean(physician_scores, axis=0)

# Existing summary scores: keep without recalculation
summary = {
    "pafa_motor_mean": "Motor Subscore",
    "pafa_nonmotor_mean": "Non-motor Subscore",
    "pafa_tot_mean": "Total PAFA Score"
}

assert all(c in df.columns for c in summary)

analysis_cols = (
    question_cols + domain_cols + list(summary)
    + ["clinical_diagnosis", "primary_parkinsonism"]
)

assert df[analysis_cols].notna().all().all()

# ============================================================
# 7. PEARSON CORRELATIONS + BH CORRECTION
# ============================================================

def calculate_matrix(features, diagnosis_column, diagnosis_map):

    codes = list(diagnosis_map)

    r = pd.DataFrame(index=features, columns=codes, dtype=float)
    p = r.copy()

    for code in codes:

        y = (df[diagnosis_column] == code).astype(int)

        for feature in features:

            x = df[feature].astype(float)

            if x.nunique() < 2 or y.nunique() < 2:
                r.loc[feature, code] = np.nan
                p.loc[feature, code] = np.nan
            else:
                r.loc[feature, code], p.loc[feature, code] = (
                    pearsonr(x, y)
                )

    # Benjamini-Hochberg correction within each heatmap
    adjusted = pd.DataFrame(
        np.nan, index=features, columns=codes
    )

    valid = p.notna().to_numpy()
    values = p.to_numpy()[valid]

    if len(values):
        corrected = multipletests(
            values, method="fdr_bh"
        )[1]

        adjusted_array = adjusted.to_numpy().copy()
        adjusted_array[valid] = corrected

        adjusted = pd.DataFrame(
            adjusted_array, index=features, columns=codes
        )

    # Left-to-right: average Pearson correlation
    order = r.mean(axis=0).sort_values(
        kind="stable"
    ).index.tolist()

    return r[order], p[order], adjusted[order], order


def annotation_matrix(r, p, adjusted):

    annotations = pd.DataFrame(
        "", index=r.index, columns=r.columns
    )

    for feature in r.index:
        for code in r.columns:

            value = r.loc[feature, code]

            if pd.isna(value):
                annotations.loc[feature, code] = "NA"
                continue

            if adjusted.loc[feature, code] < 0.05:
                stars = "**"
            elif p.loc[feature, code] < 0.05:
                stars = "*"
            else:
                stars = ""

            if abs(value) < 0.005:
                value = 0.0

            annotations.loc[feature, code] = (
                f"{value:.2f}{stars}"
            )

    return annotations

# ============================================================
# 8. EXPORT FUNCTION
# ============================================================

created_files = []

def make_heatmap(
    features,
    labels,
    diagnosis_type,
    title,
    filename,
    ylabel
):

    diagnosis_column, diagnosis_map = diagnoses[diagnosis_type]

    r, p, adjusted, order = calculate_matrix(
        features, diagnosis_column, diagnosis_map
    )

    annotations = annotation_matrix(r, p, adjusted)

    xlabels = []

    for code in order:

        n = (df[diagnosis_column] == code).sum()

        name = "\n".join(
            textwrap.wrap(
                diagnosis_map[code],
                width=20,
                break_long_words=False
            )
        )

        xlabels.append(f"{name}\n(n={n})")

    rows = len(features)
    columns = len(order)

    if rows >= 30:
        height = 20
        annotation_size = 9
        y_size = 10
    else:
        height = max(9, rows * 0.85 + 4)
        annotation_size = 13
        y_size = 12

    width = 20 if columns >= 7 else 16

    fig, ax = plt.subplots(figsize=(width, height))

    sns.heatmap(
        r,
        annot=annotations,
        fmt="",
        cmap="RdBu",
        vmin=-1,
        vmax=1,
        center=0,
        linewidths=0.7,
        linecolor="#dddddd",
        annot_kws={"fontsize": annotation_size},
        cbar_kws={
            "label": "Pearson correlation coefficient (r)",
            "shrink": 0.9
        },
        ax=ax
    )

    # Title raised and separated from heatmap
    ax.set_title(
        title,
        fontsize=21,
        fontweight="bold",
        pad=35
    )

    ax.set_xlabel(
        "Diagnosis",
        fontsize=14,
        fontweight="bold",
        labelpad=25
    )

    ax.set_ylabel(
        ylabel,
        fontsize=14,
        fontweight="bold",
        labelpad=45
    )

    # Horizontal, bold diagnosis labels
    ax.set_xticklabels(
        xlabels,
        rotation=0,
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

    ax.set_yticklabels(
        labels,
        rotation=0,
        fontsize=y_size
    )

    ax.tick_params(axis="x", length=0, pad=12)
    ax.tick_params(axis="y", length=0, pad=8)

    # Bold summary score row labels
    for label in ax.get_yticklabels():

        if label.get_text() in summary.values():
            label.set_fontweight("bold")

    # Footnote
    note = (
        "* Unadjusted p < 0.05   |   "
        "** Benjamini-Hochberg adjusted p < 0.05\n"
        "Columns ordered by mean Pearson r "
        "(most negative to most positive).\n"
        "Small diagnostic groups (n = 1-3) "
        "should be interpreted cautiously."
    )

    if ylabel == "PAFA Question":
        note += (
            "\nIndividual PAFA questions: "
            "median of three physician ratings."
        )
    else:
        note += (
            "\nDomain and summary scores: "
            "mean of three physician scores."
        )

    fig.text(
        0.5, 0.025,
        note,
        ha="center",
        va="bottom",
        fontsize=10
    )

    fig.subplots_adjust(
        left=0.29 if rows >= 30 else 0.26,
        right=0.92,
        top=0.91,
        bottom=0.22
    )

    path = os.path.join(folder, filename)

    fig.savefig(
        path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    created_files.append(filename)

    print("Created:", filename)

# ============================================================
# 9. GENERATE ALL TEN HEATMAPS
# ============================================================

for diagnosis_type in ["Clinical", "Pathological"]:

    # 1. Individual questions
    make_heatmap(
        question_cols,
        question_labels,
        diagnosis_type,
        f"PAFA Questions vs. {diagnosis_type} Diagnoses",
        f"01_PAFA_Questions_{diagnosis_type}.png",
        "PAFA Question"
    )

    # 2. Individual questions + total PAFA score
    make_heatmap(
        question_cols + ["pafa_tot_mean"],
        question_labels + ["Total PAFA Score"],
        diagnosis_type,
        f"PAFA Questions and Total Score vs. {diagnosis_type} Diagnoses",
        f"02_PAFA_Questions_Plus_Total_{diagnosis_type}.png",
        "PAFA Question"
    )

    # 3. Nine domains
    make_heatmap(
        domain_cols,
        domain_labels,
        diagnosis_type,
        f"PAFA Domains vs. {diagnosis_type} Diagnoses",
        f"03_PAFA_Domains_{diagnosis_type}.png",
        "PAFA Domain"
    )

    # 4. Nine domains + total PAFA score
    make_heatmap(
        domain_cols + ["pafa_tot_mean"],
        domain_labels + ["Total PAFA Score"],
        diagnosis_type,
        f"PAFA Domains and Total Score vs. {diagnosis_type} Diagnoses",
        f"04_PAFA_Domains_Plus_Total_{diagnosis_type}.png",
        "PAFA Domain"
    )

    # 5. Nine domains + motor, nonmotor, total
    make_heatmap(
        domain_cols + [
            "pafa_motor_mean",
            "pafa_nonmotor_mean",
            "pafa_tot_mean"
        ],
        domain_labels + [
            "Motor Subscore",
            "Non-motor Subscore",
            "Total PAFA Score"
        ],
        diagnosis_type,
        f"PAFA Domains and Summary Scores vs. {diagnosis_type} Diagnoses",
        f"05_PAFA_Domains_All_Scores_{diagnosis_type}.png",
        "PAFA Domain"
    )

# ============================================================
# 10. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 55)
print("PAFA HEATMAPS V08 — COMPLETE")
print("=" * 55)

print("Patients:", len(df))
print("PAFA questions:", len(question_cols))
print("PAFA domains:", len(domain_cols))
print("Clinical diagnoses:", len(clinical))
print("Primary pathological diagnoses:", len(pathological))

print("\nOutput folder:", folder)
print("PNG files created:", len(created_files))

for filename in created_files:
    print(filename)

assert len(created_files) == 10

print("\nSUCCESS: All 10 heatmaps exported.")


In [ ]:

# ============================================================
# PAFA HEATMAPS VERSION 09
# Exclude:
#   1. Lower motor neuron signs
#   2. Cortical sensory loss
#
# Generate 10 new PNG heatmaps at 600 DPI
# Preserve all v08 files
# ============================================================

import os

# Create a NEW output folder
folder = "PAFA_heatmaps_v09"
os.makedirs(folder, exist_ok=True)

# Reset file tracking
created_files = []

# ------------------------------------------------------------
# 1. REMOVE TWO QUESTIONS FROM HEATMAP DISPLAY
# ------------------------------------------------------------

exclude = {
    "lmn_signs_rcd_median",
    "cortical_sensory_loss_rcd_median"
}

# Preserve original question order
filtered_questions = [
    (col, label)
    for col, label in zip(question_cols, question_labels)
    if col not in exclude
]

question_cols_v09 = [
    col for col, label in filtered_questions
]

question_labels_v09 = [
    label for col, label in filtered_questions
]

# Verify removal
assert len(question_cols_v09) == 34
assert not any(
    col in question_cols_v09 for col in exclude
)

print("PAFA questions retained:", len(question_cols_v09))
print("PAFA questions excluded:", len(exclude))

# ------------------------------------------------------------
# 2. CREATE ALL 10 REVISED HEATMAPS
# ------------------------------------------------------------

for diagnosis_type in ["Clinical", "Pathological"]:

    # FIGURE 1: Individual PAFA questions
    make_heatmap(
        question_cols_v09,
        question_labels_v09,
        diagnosis_type,
        f"PAFA Questions vs. {diagnosis_type} Diagnoses",
        f"01_PAFA_Questions_{diagnosis_type}.png",
        "PAFA Question"
    )

    # FIGURE 2: Questions + total PAFA score
    make_heatmap(
        question_cols_v09 + ["pafa_tot_mean"],
        question_labels_v09 + ["Total PAFA Score"],
        diagnosis_type,
        f"PAFA Questions and Total Score vs. {diagnosis_type} Diagnoses",
        f"02_PAFA_Questions_Plus_Total_{diagnosis_type}.png",
        "PAFA Question"
    )

    # FIGURE 3: Nine PAFA domains
    make_heatmap(
        domain_cols,
        domain_labels,
        diagnosis_type,
        f"PAFA Domains vs. {diagnosis_type} Diagnoses",
        f"03_PAFA_Domains_{diagnosis_type}.png",
        "PAFA Domain"
    )

    # FIGURE 4: Nine domains + total score
    make_heatmap(
        domain_cols + ["pafa_tot_mean"],
        domain_labels + ["Total PAFA Score"],
        diagnosis_type,
        f"PAFA Domains and Total Score vs. {diagnosis_type} Diagnoses",
        f"04_PAFA_Domains_Plus_Total_{diagnosis_type}.png",
        "PAFA Domain"
    )

    # FIGURE 5: Nine domains + all summary scores
    make_heatmap(
        domain_cols + [
            "pafa_motor_mean",
            "pafa_nonmotor_mean",
            "pafa_tot_mean"
        ],
        domain_labels + [
            "Motor Subscore",
            "Non-motor Subscore",
            "Total PAFA Score"
        ],
        diagnosis_type,
        f"PAFA Domains and Summary Scores vs. {diagnosis_type} Diagnoses",
        f"05_PAFA_Domains_All_Scores_{diagnosis_type}.png",
        "PAFA Domain"
    )

# ------------------------------------------------------------
# 3. VERIFY OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("PAFA HEATMAPS VERSION 09 — COMPLETE")
print("=" * 55)

print("\nPatients:", len(df))
print("Individual questions displayed:", len(question_cols_v09))
print("PAFA domains:", len(domain_cols))
print("Clinical diagnostic groups:", len(clinical))
print("Primary pathological groups:", len(pathological))

print("\nOutput folder:", folder)
print("PNG files created:", len(created_files))

for filename in created_files:
    print(filename)

assert len(created_files) == 10

print("\nSUCCESS: All 10 revised heatmaps exported.")
print("Previous v08 figures remain unchanged.")


In [ ]:
import sys

print("Python version:", sys.version.split()[0])